In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
from folium.plugins import MarkerCluster
from sodapy import Socrata


In [ ]:
# Load city-owned land data
df = pd.read_csv("City-Owned_Land_Inventory_20250207.csv")

# Clean and standardize PIN format
df.rename(columns={'PIN': 'pin'}, inplace=True)
df['pin'] = df['pin'].str.replace('-', '', regex=True)
pin_values = df['pin'].dropna().unique().tolist()


In [ ]:
city_shape = gpd.read_file("Chicago_City_Limits.shp")


In [ ]:
client = Socrata("datacatalog.cookcountyil.gov", None)

chunk_size = 100
filtered_results = []

for i in range(0, len(pin_values), chunk_size):
    pin_chunk = pin_values[i:i + chunk_size]
    pin_filter = " OR ".join([f"pin='{p}'" for p in pin_chunk])

    try:
        results = client.get("nj4t-kc8j", where=pin_filter, limit=50000)
        if results:
            filtered_results.append(pd.DataFrame.from_records(results))
        print(f"Fetched {i + len(pin_chunk)} records...")
    except Exception as e:
        print(f"Error fetching batch {i}-{i+chunk_size}: {e}")

results_df = pd.concat(filtered_results, ignore_index=True) if filtered_results else pd.DataFrame()
if not results_df.empty:
    results_df['pin'] = results_df['pin'].astype(str)


In [ ]:
merged_df = df.merge(results_df, on='pin', how='inner')
df_latest = merged_df.loc[merged_df.groupby('pin')['year'].idxmax()]
df_latest.head()
